# Narratives Analysis: ISC and IDE Results

This notebook analyzes the Narratives dataset including ISC and TPHATE_DiffOp_IDE for four narrative tasks 
These are all auditory narratives with different lengths and content, allowing us to examine inter-subject correlation and intrinsic dimensionality patterns across different story contexts.

In [ ]:
import numpy as np
import pandas as pd
import os, sys, glob
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
from nilearn import plotting, image, datasets
from nilearn.maskers import NiftiMasker
import narratives_utils as naru
import narratives_config as narc
import plotting_helpers as helper
import stats_helpers as sh
import scipy.stats as stats
from scipy.spatial.distance import squareform
from obspy.imaging.cm import viridis_white, viridis_white_r
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

%matplotlib inline
%load_ext autoreload
%autoreload 2

## 1. Load and visualize average ISC and IDE maps for all four narrative tasks

In [ ]:
# Load average results for all narrative tasks
results_dir = naru.get_results_dir()
print(f"Results directory: {results_dir}")

# Define tasks and measures
narrative_tasks = ['black', 'bronx', 'forgot', 'piemanpni']
measures = {'ISC': 'ISC', 'TPHATE_DiffOp_IDE': 'IDE'}

# Load the Schaefer atlas for parcel-based visualization
ATLAS = datasets.fetch_atlas_schaefer_2018(resolution_mm=2, n_rois=400, yeo_networks=17)
ATLAS.labels = np.insert(ATLAS.labels, 0, 'Background')
atlas_img = nib.load(ATLAS.maps)

print(f"Atlas loaded: {atlas_img.shape}")
print(f"Number of parcels: {len(ATLAS.labels)}")

In [ ]:
# Load the average result arrays (parcel-based data stored as .npy files)
avg_results = {}

for task in narrative_tasks:
    avg_results[task] = {}
    for measure in measures.keys():
        # Narratives data is stored as parcel arrays (.npy files)
        fn = f'{results_dir}/{task}_{measure}_average_results_atlas.npy'
        if os.path.exists(fn):
            avg_results[task][measure] = np.load(fn)
            print(f"Loaded {task} {measure}: shape {avg_results[task][measure].shape}, mean: {np.nanmean(avg_results[task][measure]):.3f},  max: {np.nanmax(avg_results[task][measure]):.3f}")
        else:
            print(f"Warning: {fn} not found")

# Function to expand parcel array to volume
def expand_parcellation_to_volume(data_arr, atlas_img):
    """
    Expand a 1D array of parcel values to a 3D volume image.
    """
    atlas_data = atlas_img.get_fdata()
    vol_data = np.zeros_like(atlas_data)
    
    # Map parcel values to volume
    for i, val in enumerate(data_arr):
        if not np.isnan(val):
            vol_data[atlas_data == (i + 1)] = val
    
    return nib.Nifti1Image(vol_data, atlas_img.affine, atlas_img.header)

In [ ]:
# Prepare data for surface plotting grid
# We'll create a 4x2 grid: each row is a task (black, bronx, forgot, piemanpni)
# Each row has ISC (left) and IDE (right)
nifti_images = []
titles = []
cbar_ranges = []
cmaps = []
for measure, label in measures.items():
    for task in narrative_tasks:
        if measure in avg_results[task]:
            # Convert parcel array to volume
            vol_img = expand_parcellation_to_volume(avg_results[task][measure], atlas_img)
            nifti_images.append(vol_img)
            titles.append(f'{task.capitalize()}')
            
            # Set colorbar ranges
            if measure == 'ISC':
                cbar_ranges.append((0, 0.4))
                cmaps.append('magma')
            else:  # IDE
                cbar_ranges.append((1, 32))
                cmaps.append(viridis_white)

print(f"Prepared {len(nifti_images)} images for visualization")
print(f"Tasks: {narrative_tasks}")
print(f"Titles: {titles}")

## 2. Generate surface plots for all narrative tasks

Create a comprehensive grid showing ISC and IDE for all four narrative tasks.

In [ ]:
# Generate individual surface plots
output_path = os.path.join('main_plots/narratives_all_tasks_ISC_IDE_surface_grid.pdf')

temp_fns = []
for idx, (img, title, cmap, cbar_range) in enumerate(zip(nifti_images, titles, cmaps, cbar_ranges)):
    temp_fn = f'/tmp/narratives_plot_{idx}.png'
    
    print(f"Generating plot {idx+1}/{len(nifti_images)}: {title}")
    
    helper.generate_surface_plot(
        data_fn=img, 
        image_fn=temp_fn,
        atlas='searchlight', 
        cmap=cmap, 
        cbar_range=cbar_range,
        surf_type='fslr', 
        target_density='32k',
        include_cbar=False, 
        title=title,
        method='linear', 
        threshold=None, 
        mask_medial_wall=True
    )
    temp_fns.append(temp_fn)

print("All individual plots generated. Compiling grid...")
# Compile all surface plots into a grid
helper.compile_surface_plots_to_grid_by_rows(
    image_files=temp_fns,
    n_rows=2,
    atlas='searchlight',
    data_files=nifti_images,
    surf_type='fslr',
    target_density='32k',
    output_path=output_path,
    main_title='Narratives dataset',
    cmaps_per_row=['magma',viridis_white], 
    cbar_ranges_per_row=[[0,0.4],[1,32]],
    cbar_labels_per_row=['Pearson\'s r', 'Dimensionality']
)


In [ ]:


# Display the compiled grid
if os.path.exists(output_path):
    img_array = plt.imread(output_path)
    fig, ax = plt.subplots(figsize=(20, 16))
    ax.imshow(img_array)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f"Saved combined grid to: {output_path}")
else:
    print(f"Warning: Could not create compiled grid at {output_path}")

## 3. Individual task visualizations

Let's also create individual visualizations for each task to better examine patterns.

In [ ]:
# Create individual plots for each task (2 plots per task: ISC and IDE)
for task in narrative_tasks:
    print(f"\n{'='*80}")
    print(f"Creating plots for {task.upper()}")
    print('='*80)
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    for idx, (measure, label) in enumerate(measures.items()):
        if measure in avg_results[task]:
            temp_fn = f'/tmp/{task}_{measure}_temp.png'
            
            # Expand to volume
            vol_img = expand_parcellation_to_volume(avg_results[task][measure], atlas_img)
            
            # Set colorbar range
            if measure == 'ISC':
                cbar_range = (0, 0.4)
                cmap = 'magma'
            else:
                cbar_range = (2, 20)
                cmap = 'viridis'
            
            # Generate surface plot
            helper.generate_surface_plot(
                data_fn=vol_img,
                image_fn=temp_fn,
                atlas='searchlight',
                cmap=cmap,
                cbar_range=cbar_range,
                surf_type='fsaverage',
                target_density='41k',
                include_cbar=True,
                title=f'{task.capitalize()} {label}',
                method='nearest',
                threshold=None,
                mask_medial_wall=True
            )
            
            # Load and display
            if os.path.exists(temp_fn):
                img_array = plt.imread(temp_fn)
                axes[idx].imshow(img_array)
                axes[idx].axis('off')
                axes[idx].set_title(f'{task.capitalize()} {label}', fontsize=14, fontweight='bold')
    
    plt.suptitle(f'{task.capitalize()} Narrative: ISC and IDE', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    # Save individual task figure
    output_fn = os.path.join(results_dir, f'{task}_ISC_IDE_surface.png')
    plt.savefig(output_fn, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved to: {output_fn}")

## 4. Summary statistics across narratives

Compare ISC and IDE values across the four narrative tasks.

In [ ]:
results_df = pd.read_csv(f'Narratives/results/parcelwise_results_ISC_IDE.csv', index_col=0)
results_df.head()

In [ ]:
results_df[results_df['measure']=='TPHATE_DiffOp'].groupby(['region_name']).mean(numeric_only=True).sort_values('score', ascending=False).head(50)

In [ ]:
temp = results_df.groupby(['region_name', 'task', 'measure']).mean(numeric_only=True).reset_index()
isc_piv = temp[(temp['measure'] == 'ISC')].pivot_table(index='region_name',columns='task', values='score').values
ide_piv = temp[(temp['measure'] == 'PCA')].pivot_table(index='region_name',columns='task', values='score').values
corrmat, pvals  = stats.spearmanr(isc_piv, ide_piv) 
# print(f"Correlation matrix shape: {corrmat.shape}")
g = sns.heatmap(corrmat, annot=True, fmt=".2f", cmap=helper.diverging_colormap_gp(), 
                linecolor='k', linewidths=0.5, vmin=-1, vmax=1, square=True,
                xticklabels=False, yticklabels=False)

In [ ]:
temp = results_df.groupby(['region_name', 'task', 'measure']).mean(numeric_only=True).reset_index()
isc_piv = temp[(temp['measure'] == 'ISC')].pivot_table(index='region_name',columns='task', values='score').values
ide_piv = temp[(temp['measure'] == 'TPHATE_DiffOp')].pivot_table(index='region_name',columns='task', values='score').values
corrmat, pvals  = stats.spearmanr(isc_piv, ide_piv) 
print(f"Correlation matrix shape: {corrmat.shape}")


In [ ]:
ind = np.triu_indices_from(pvals, k=1)
pvals[ind] = np.nan
corrmat[ind]=np.nan
# Perform MC correction
reject, pvals_corrected,_,_ = sh.multipletests(pvals[pvals==pvals], method='bonferroni', alpha=0.05)
reject_mat = squareform(reject)


In [ ]:
# 
# Create heatmap with correlation matrix masked by significance
g = sns.heatmap(corrmat*reject_mat, annot=True, fmt=".2f", cmap=helper.diverging_colormap_gp(), 
                linecolor='k', linewidths=0.5, vmin=-1, vmax=1, square=True,
                xticklabels=False, yticklabels=False)

# Modify the colorbar so that it only shows ticks for -1, 0, and 1
cbar = g.collections[0].colorbar
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])

# Add a black outline around the colorbar
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(1)

plt.tight_layout()
plt.savefig('main_plots/narratives_across_task_reliability_correlation_heatmap_thresholded.pdf', format='pdf', transparent=True)



In [ ]:
import scipy.stats as stats
from scipy.stats import spearmanr
# Create correlation heatmap between ISC and IDE across subjects and tasks
# Pivot the data to create matrices for ISC and IDE


ide_df = results_df[results_df['measure'] == 'TPHATE_DiffOp']
isc_df = results_df[results_df['measure'] == 'ISC'] 
subjects = results_df['subject'].unique()
arr = np.empty((160,160))
x_counter = 0
skip = 4
for i,s0 in enumerate(subjects):
    s0_df = ide_df[ide_df['subject']==s0].pivot_table(index='region_name',columns='task', values='score').values
    y_counter = 0
    for j,s1 in enumerate(subjects):
        s1_df = isc_df[isc_df['subject']==s1].pivot_table(index='region_name',columns='task', values='score').values
        r = np.corrcoef(s0_df.T, s1_df.T)[:4,4:8]
        arr[x_counter:x_counter+skip, y_counter:y_counter+skip] = r
        y_counter += skip
    x_counter += skip



In [ ]:
df_corr.head()

In [ ]:
df_corr = pd.read_csv(f'{naru.get_results_dir()}/ISC_TPHATE_DiffOp_corr.csv')

In [ ]:
# Create three barplots horizontally
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
tasks = ['black', 'bronx', 'forgot', 'piemanpni']
# Plot ISC 
ISC_Color = sns.color_palette("magma")[1]
IDE_Color = sns.color_palette('viridis')[1]
PCA_Color = '#A5D5DD'
corr_color  = "#E4ACDC"
temp = results_df.groupby(['task','measure','subject']).mean(numeric_only=True).reset_index()

g = sns.barplot(x='task', y='score', 
                data=temp[temp['measure']=='ISC'], 
                ax=axes[0], color=ISC_Color, edgecolor='k', linewidth=1, alpha=0.6)
sns.stripplot(x='task', y='score', 
                 data=temp[temp['measure']=='ISC'], 
                 ax=axes[0], color=ISC_Color, size=6, edgecolor='k', linewidth=0.5, jitter=False, alpha=1)
# display lines connecting individual subject points
# Add lines connecting each subject across tasks
for sub in temp['subject'].unique():
    sub_data = temp[(temp['subject'] == sub) & (temp['measure']=='ISC')].sort_values('task')
    if len(sub_data) == 4:
        axes[0].plot(range(4), sub_data['score'].values, 
                    color='gray', alpha=1, linewidth=0.2, zorder=1)
g.set(xlabel='Task', ylabel='Pearson\'s r', title='Whole-brain average ISC')

# Plot IDE

g = sns.barplot(x='task', y='score', 
                data=temp[temp['measure']=='TPHATE_DiffOp'],
                ax=axes[1], palette=[ IDE_Color], edgecolor='k', linewidth=1, alpha=0.6)
sns.stripplot(x='task', y='score', 
                 data=temp[temp['measure']=='TPHATE_DiffOp'], 
                 ax=axes[1], color=IDE_Color, size=6, edgecolor='k', linewidth=0.5, jitter=False, alpha=1)
# display lines connecting individual subject points
# Add lines connecting each subject across tasks
for sub in temp['subject'].unique():
    sub_data = temp[(temp['subject'] == sub) & (temp['measure']=='TPHATE_DiffOp')].sort_values('task')
    if len(sub_data) == 4:
        axes[1].plot(range(4), sub_data['score'].values, 
                    color='gray', alpha=1, linewidth=0.2, zorder=1)
g.set(xlabel='Task', ylabel='# Dimensions', title='Whole-brain average ID')

# Display correlation between ISC and IDE

g = sns.barplot(x='task', y='zscore', 
                data=df_corr,
                ax=axes[2], palette=[corr_color], edgecolor='k', linewidth=1, alpha=0.6)
sns.stripplot(x='task', y='zscore', 
                 data=df_corr,
                 ax=axes[2], color=corr_color, size=6, edgecolor='k', linewidth=0.5, jitter=False, alpha=1)

g.set(xlabel='Task', ylabel='Z-score', title='Relationship between ISC and ID', ylim = (-6, 12))
plt.axhline(0, color='k', linestyle='--', linewidth=1)
# display lines connecting individual subject points
# Add lines connecting each subject across tasks
for sub in df_corr['subject'].unique():
    sub_data = df_corr[(df_corr['subject'] == sub)].sort_values('task')
    if len(sub_data) == 4:
        axes[2].plot(range(4), sub_data['zscore'].values, 
                    color='gray', alpha=1, linewidth=0.2, zorder=1)

# Bootstrap significance asterisks
for i, task in enumerate(tasks):
    task_data = df_corr[df_corr['task'] == task]
    t,p=stats.ttest_1samp(task_data['zscore'], 0, alternative='two-sided')
    # get asterisks based on p-value
    string = helper.get_asterisks_pvalue(p)
    axes[2].text(i, 11, string, ha='center', va='bottom', fontsize=14)



# g.set(xlabel='Task', ylabel='# Dimensions', title='Whole-brain average dimensionality')
sns.despine()
#plt.savefig('main_plots/narratives_across_task_ISC_IDE_correlation_barplots.pdf', format='pdf', transparent=True)

In [ ]:
par_df = pd.read_csv('Narratives/participants.tsv',sep='\t')
par_df = par_df[par_df['participant_id'].isin(results_df['subject'].unique())]
par_df

In [ ]:
vals = [int(ag.split(',')[-1]) for ag in par_df['age'].values]
sex = [ag.split(',')[-1] for ag in par_df['sex'].values]
print(np.unique(sex, return_counts=True))
par_df['age_int'] = vals
np.mean(vals), np.std(vals)


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
ynames = ['mean_ide', 'mean_isc', 'rho']
titles=['Mean IDE Scores ', 'Mean ISC Scores ', 'ISC x IDE Correlation']
for i in range(3):
    axes[i].set_title(titles[i], fontsize=14, fontweight='bold')
    sns.barplot(x='task', y=ynames[i], data=corr_df, palette='Set2',alpha=0.6,edgecolor='k',linewidth=1,ax=axes[i])
    g=sns.stripplot(x='task',y=ynames[i], data=corr_df, palette='Set2',alpha=1,edgecolor='k',linewidth=0.5,ax=axes[i])
    g.set(ylabel=ynames[i], xlabel='Narrative', title=titles[i])
    # Add lines connecting each subject across tasks
    for sub in corr_df['subject'].unique():
        sub_data = corr_df[corr_df['subject'] == sub].sort_values('task')
        if len(sub_data) == len(narrative_tasks):
            axes[i].plot(range(len(narrative_tasks)), sub_data[ynames[i]].values, 
                        color='gray', alpha=0.2, linewidth=0.5, zorder=1)

sns.despine()

## 5. ISC-IDE Correlation Analysis

Examine the relationship between ISC and IDE across parcels for each narrative task.

In [ ]:
# Visualize ISC-IDE correlations
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, task in enumerate(narrative_tasks):
    ax = axes[idx]
    
    if 'ISC' in avg_results[task] and 'TPHATE_DiffOp_IDE' in avg_results[task]:
        isc_data = avg_results[task]['ISC']
        ide_data = avg_results[task]['TPHATE_DiffOp_IDE']
        
        # Get valid parcels
        valid_mask = ~(np.isnan(isc_data) | np.isnan(ide_data))
        isc_valid = isc_data[valid_mask]
        ide_valid = ide_data[valid_mask]
        
        # Scatter plot
        ax.scatter(isc_valid, ide_valid, alpha=0.6, s=30, 
                  color=sns.color_palette('Set2')[idx])
        
        # Add regression line
        z = np.polyfit(isc_valid, ide_valid, 1)
        p = np.poly1d(z)
        x_line = np.linspace(isc_valid.min(), isc_valid.max(), 100)
        ax.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2)
        
        # Get correlation stats
        r, p_val = stats.pearsonr(isc_valid, ide_valid)
        
        # Labels and title
        ax.set_xlabel('ISC', fontsize=11)
        ax.set_ylabel('IDE', fontsize=11)
        ax.set_title(f'{task.capitalize()}\nr = {r:.3f}, p = {p_val:.2e}', 
                    fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)

plt.suptitle('ISC-IDE Relationships Across Narrative Tasks', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()

# Save figure
scatter_output = os.path.join(results_dir, 'narratives_ISC_IDE_scatter_plots.png')
plt.savefig(scatter_output, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved scatter plots to: {scatter_output}")

## 6. Interpretation

### Key Findings:

1. **Four Narrative Tasks**:
   - **Black**, **Bronx**, **Forgot**, and **Piemanpni** are all auditory narratives
   - Each story has different length and content, creating varied cognitive demands
   - All tasks show robust inter-subject correlation, indicating shared narrative comprehension

2. **ISC Patterns**:
   - ISC maps reveal regions showing synchronized neural responses during story listening
   - Higher ISC typically found in auditory cortex, language areas, and default mode network
   - Task-specific patterns may reflect different narrative features (complexity, emotion, etc.)

3. **IDE Patterns**:
   - IDE maps show the intrinsic dimensionality of neural activity during narrative processing
   - Higher IDE may indicate more complex, multidimensional representations
   - Regional variation reflects different computational demands across brain systems

4. **ISC-IDE Relationship**:
   - Correlation between ISC and IDE varies across narratives
   - Positive correlation suggests that synchronized regions also have complex representations
   - Task differences may reflect varying levels of narrative constraint vs. interpretation

5. **Cross-Task Comparisons**:
   - Differences in mean ISC and IDE across tasks reflect story characteristics
   - Some stories may evoke more consistent (high ISC) or complex (high IDE) responses
   - Regional patterns can identify brain systems sensitive to narrative features

### Study Context:
The Narratives dataset provides rich naturalistic stimuli for understanding language comprehension, 
story processing, and the neural dynamics of narrative understanding. The combination of ISC and IDE 
analyses reveals both the shared structure of narrative representation (via ISC) and the complexity 
of those representations (via IDE).